# Week 11: Advanced Specialized Machine Learning
## Notebook 12: Genetic Algorithms for Combinatorial Feature Selection

### 1. Conceptual Intuition & Theoretical Background
Selecting the optimal subset of $k$ features from $D$ candidates is an NP-hard combinatorial problem with a search space of $2^D$ configurations. Greedy approaches (like Forward Selection or RFE) can get trapped in sub-optimal local minima.

**Genetic Algorithms (GA, Holland, 1975):**
Simulates natural selection and evolutionary biology to navigate combinatorial spaces:
1. **Chromosome Representation:** A binary bitmask vector $c \in \{0, 1\}^D$ where $1$ denotes feature inclusion.
2. **Fitness Function:** Model evaluation metric (e.g. validation $R^2$ or accuracy) penalized by feature count:
$$\text{Fitness}(c) = \text{ValidationScore}(c) - \gamma \cdot \frac{\sum c_i}{D}$$
3. **Selection (Roulette Wheel / Tournament):** Selects high-fitness parent chromosomes with higher probability.
4. **Crossover:** Exchanges genetic substrings between paired chromosomes to combine promising feature combinations.
5. **Mutation:** Flips individual bitmask genes with small probability $p_m$ to maintain population diversity.

---

### 2. Scratch Python Implementation: Genetic Feature Selector

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

class GeneticFeatureSelectorScratch:
    def __init__(self, n_features, pop_size=20, n_generations=15, p_crossover=0.8, p_mutation=0.05):
        self.n_feat = n_features
        self.pop_size = pop_size
        self.n_gen = n_generations
        self.p_cross = p_crossover
        self.p_mut = p_mutation
        self.best_chromosome = None
        self.best_fitness = -np.inf
        
    def _fitness(self, chromosome, X, y):
        selected = np.where(chromosome == 1)[0]
        if len(selected) == 0:
            return -999.0
        # 3-Fold CV R^2 score with parsimony penalty
        model = Ridge(alpha=1.0)
        scores = cross_val_score(model, X[:, selected], y, cv=3, scoring='r2')
        # Penalize excessive feature count (Occam's razor)
        parsimony_penalty = 0.02 * (len(selected) / self.n_feat)
        return float(np.mean(scores)) - parsimony_penalty
        
    def fit(self, X, y):
        # 1. Initialize random binary population
        np.random.seed(42)
        population = np.random.randint(0, 2, size=(self.pop_size, self.n_feat))
        
        for gen in range(self.n_gen):
            fitness_scores = np.array([self._fitness(ind, X, y) for ind in population])
            
            # Track best individual
            best_idx = np.argmax(fitness_scores)
            if fitness_scores[best_idx] > self.best_fitness:
                self.best_fitness = fitness_scores[best_idx]
                self.best_chromosome = population[best_idx].copy()
                
            # 2. Tournament Selection
            new_population = []
            for _ in range(self.pop_size // 2):
                # Sample 2 parents via tournament
                t1 = np.random.choice(self.pop_size, size=3, replace=False)
                p1 = population[t1[np.argmax(fitness_scores[t1])]]
                
                t2 = np.random.choice(self.pop_size, size=3, replace=False)
                p2 = population[t2[np.argmax(fitness_scores[t2])]]
                
                # 3. Crossover (Single-Point)
                c1, c2 = p1.copy(), p2.copy()
                if np.random.rand() < self.p_cross:
                    pt = np.random.randint(1, self.n_feat)
                    c1[pt:], c2[pt:] = p2[pt:], p1[pt:]
                    
                # 4. Mutation
                mut_mask1 = np.random.rand(self.n_feat) < self.p_mut
                c1[mut_mask1] = 1 - c1[mut_mask1]
                mut_mask2 = np.random.rand(self.n_feat) < self.p_mut
                c2[mut_mask2] = 1 - c2[mut_mask2]
                
                new_population.extend([c1, c2])
                
            population = np.array(new_population)
            
        return self

# Benchmark Dataset: 15 Features (Only 3 are truly predictive!)
np.random.seed(42)
N = 150
D = 15
X_data = np.random.normal(0, 1, (N, D))
# Target only depends on feature 0, 1, and 2
y_data = 3.0 * X_data[:, 0] - 2.5 * X_data[:, 1] + 1.8 * X_data[:, 2] + np.random.normal(0, 0.2, N)

ga = GeneticFeatureSelectorScratch(n_features=D, pop_size=20, n_generations=12)
ga.fit(X_data, y_data)

selected_features = np.where(ga.best_chromosome == 1)[0]
print("--- Genetic Algorithm Feature Selection ---")
print(f"Total Candidate Features:      {D}")
print(f"True Signal Features:          [0, 1, 2]")
print(f"GA Selected Features:          {selected_features.tolist()}")
print(f"Best Discovered Fitness (R2):  {ga.best_fitness:.4f}")

### 3. Step-by-Step Code Walkthrough
1. **Binary Chromosome Encoding:** Encodes feature configurations as binary bitmasks.
2. **Tournament Selection:** Selects top individuals from randomized subsets, maintaining evolutionary selection pressure.
3. **Crossover & Mutation:** Explores non-linear feature combinations while maintaining population diversity.

---

### 4. Industry Applications & Trade-offs
- **Quantitative Trading & Alpha Search:** Exploring combinations of hundreds of technical indicators.
- **Trade-off:** High computational cost because every candidate chromosome requires training and evaluating a cross-validation model.

---

### 5. Top Interview Questions & Answers
**Q1: How do Genetic Algorithms prevent premature convergence to sub-optimal local minima?**
> **Answer:** Through the mutation operator, which randomly flips genes with probability $p_m$. This introduces new genetic material into the population and allows the search to escape local traps.

**Q2: What is the purpose of the parsimony penalty in the fitness function?**
> **Answer:** It penalizes chromosomes that select excessive features, enforcing Occam's razor and preventing the algorithm from retaining uninformative noise columns.